# 01b — NDVI phenology metrics

Computes three per-pixel NDVI phenology metrics from the 5-band monthly NDVI composite (June–October) and writes them as a 3-band raster included in the classification feature stack.

Band order in the output (matches the manuscript feature table):

| Band | Metric | Definition | Feature ID |
|------|--------|------------|------------|
| B1 | NDVI seasonal range | per-pixel (max − min) across the 5 months | F56 |
| B2 | NDVI summer mean | mean of July & August (input bands 2–3) | F57 |
| B3 | NDVI autumn decline | August − October (input bands 3 − 5) | F58 |

In the deployed model, `NDVI_phenology_metrics_B1` (seasonal range, F56) is the phenology feature retained by selection (notebook 04). Per-pixel max and min are computed only as intermediates for the seasonal range and are not themselves stacked as features.

Input months are ordered June, July, August, September, October (bands 1–5).

_Manuscript: Methods — NDVI phenology features (F56–F58)._

In [ ]:
import rasterio
import numpy as np

DATA_DIR = "DATA_DIR"  # <-- set project root

# 5-band monthly NDVI composite (Jun, Jul, Aug, Sep, Oct), per mosaic/year
input_stack  = f"{DATA_DIR}/Rasters/Sentinel2_NDVI_Stack_2024_Filled.tif"
output_stack = f"{DATA_DIR}/Rasters/NDVI_phenology_metrics.tif"

# === Load NDVI stack (shape: 5 x H x W; band order Jun..Oct) ===
with rasterio.open(input_stack) as src:
    ndvi_stack = src.read().astype(np.float32)
    profile = src.profile.copy()

# === Intermediates (max/min used only to derive the seasonal range) ===
ndvi_max = np.nanmax(ndvi_stack, axis=0)
ndvi_min = np.nanmin(ndvi_stack, axis=0)

# === Three stacked phenology features (order matches feature table F56-F58) ===
ndvi_seasonal_range = ndvi_max - ndvi_min                 # B1 / F56
ndvi_summer_mean    = np.nanmean(ndvi_stack[1:3], axis=0) # B2 / F57: Jul & Aug (bands 2-3)
ndvi_autumn_decline = ndvi_stack[2] - ndvi_stack[4]      # B3 / F58: Aug - Oct (band 3 - 5)

phenology_stack = np.stack([
    ndvi_seasonal_range,   # B1 -> NDVI_phenology_metrics_B1 (used in final model, F56)
    ndvi_summer_mean,      # B2 (F57)
    ndvi_autumn_decline,   # B3 (F58)
])

profile.update({"count": 3, "dtype": "float32", "compress": "lzw", "BIGTIFF": "YES"})

with rasterio.open(output_stack, "w", **profile) as dst:
    dst.write(phenology_stack)
    dst.set_band_description(1, "ndvi_seasonal_range")
    dst.set_band_description(2, "ndvi_summer_mean")
    dst.set_band_description(3, "ndvi_autumn_decline")

print("Saved NDVI phenology metrics (3 bands) ->", output_stack)
